# tiny-voca-lg: train the Luganda model on Kaggle (two T4s)

Fine-tunes the English [tiny-voca](https://github.com/n4ze3m/tiny-voca) (4.5M parameters) on
`CraneAILabs/waxal-lug-clean` with one female voice (F2) and one male voice (M2), using the code in
[henryhale/tiny-voca-lg](https://github.com/henryhale/tiny-voca-lg). The Luganda front end, `--init-weights`,
multi-GPU training and the data scripts are all in that repository, so nothing is patched here.
The settings are the ones the released weights were trained with.

**Everything lives in `/kaggle/working/`, and every step resumes.** A finished step leaves a marker in
`/kaggle/working/stages/` and is skipped next time; an interrupted step continues where it stopped
(training from its last save, at most 500 steps back). To pick up a stopped run:

- **Interactive session:** Settings → *Persistence* → **Files only**, then *Run All* again.
- **Saved version:** *Add Input* → your previous version's output, set `RESUME_FROM` in cell 1, then *Save & Run All*.

Settings: Accelerator **GPU T4 ×2**, Internet **on**.

| Step | Cell | Code | Uses | Time |
|---|---|---|---|---|
| 1. Setup and downloads | 1–3 | | | ~5 min |
| 2. Export the two speakers at 24 kHz | 4 | `scripts/export_waxal.py` | CPU | ~5 min |
| 3. Cut long clips into 4–18 s chunks | 5 | `scripts/split_clips.py` | both GPUs, one speaker each | ~5 min |
| 4. Choose the audio, acoustic features | 6 | `scripts/make_manifests.py`, `tiny_voca.prepare` | all CPU cores | ~15–25 min |
| 5. Phoneme alignment | 7 | `tiny_voca.annotate` | GPU 0 | ~5 min |
| 6. Training: 4,000 steps | 8 | `tiny_voca.train` | both GPUs, one model | ~1 hour |
| 7. Listen to checkpoints, package | 9–10 | `scripts/export_checkpoint.py`, `tiny_voca.export`, `scripts/export_gguf.py` | | ~10 min |

In [ ]:
# 1. Settings and folders
import os, sys, json, time, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/henryhale/tiny-voca-lg"
REPO_REF = "master"              # a branch or tag; pin a tag to repeat a run exactly
FEMALE, MALE = "5", "1"          # Waxal speaker_id: 5 (female) and 1 (male) have the most audio
MINUTES_PER_VOICE = 100          # training audio per voice after cutting
STEPS = 4000                     # one step = one batch per GPU; about 80 passes over the data
LEARNING_RATE = 2e-4             # doubled for the doubled batch (two GPUs)
BATCH_FRAMES = 9000              # per GPU; drops to 6000 automatically if a T4 runs out of memory
MAX_TRAIN_SECONDS = 3 * 3600     # per session; training resumes from its last save next time
RESUME_FROM = None               # e.g. "/kaggle/input/tiny-voca-lg-train" (a previous version's output)

WORK = Path("/kaggle/working")
REPO = WORK / "tiny-voca-lg"     # code
OUT = WORK / "luganda"           # samples, final model
DATA = WORK / "data"             # clips, chunks, manifests
FEATURES = WORK / "features"     # acoustic features
ANN = WORK / "annotations"       # phonemes and durations
RUN = WORK / "run"               # training checkpoints
LOGS = WORK / "logs"
STAGES = WORK / "stages"         # one marker per finished step
voices = {FEMALE: "F2", MALE: "M2"}

if RESUME_FROM and not any(STAGES.glob("*.done")):
    print(f"copying the previous run from {RESUME_FROM} ...")
    shutil.copytree(RESUME_FROM, WORK, dirs_exist_ok=True, ignore=shutil.ignore_patterns("__notebook__*", "*.html"))
for d in (OUT, DATA, FEATURES, LOGS, STAGES):
    d.mkdir(parents=True, exist_ok=True)

# The data steps depend on these settings: refuse to mix runs made with different ones.
settings = {"female": FEMALE, "male": MALE, "minutes_per_voice": MINUTES_PER_VOICE}
settings_file = STAGES / "settings.json"
if settings_file.exists() and json.loads(settings_file.read_text()) != settings:
    raise SystemExit(f"Settings changed since the saved run ({settings_file.read_text()}). "
                     "Delete /kaggle/working/stages, data, features, annotations and run to start over.")
settings_file.write_text(json.dumps(settings))

os.environ["PYTHONPATH"] = str(REPO / "src")
os.environ["TINY_VOCA_FRONTEND"] = "lug"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def done(stage):
    return (STAGES / f"{stage}.done").exists()

def mark(stage, **info):
    (STAGES / f"{stage}.done").write_text(json.dumps(info) + "\n")
    print(f"✓ {stage} finished: {info}")

def skip(stage):
    if done(stage):
        print(f"✓ {stage}: already done, skipping ({(STAGES / f'{stage}.done').read_text().strip()})")
        return True
    return False

def read_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding="utf-8").splitlines() if l.strip()]

def write_jsonl(path, rows):
    Path(path).write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), encoding="utf-8")

def last_line(text):
    lines = [l for l in text.splitlines() if l.strip()]
    return lines[-1] if lines else ""

def start(name, cmd, gpu=None, single_thread=False):
    log = LOGS / f"{name}.log"
    env = dict(os.environ)
    if gpu is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    if single_thread:  # one job per core: stop numpy/numba/torch from oversubscribing the CPU
        env.update({k: "1" for k in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMBA_NUM_THREADS")})
    return subprocess.Popen(cmd, env=env, stdout=open(log, "a"), stderr=subprocess.STDOUT), log

def watch(jobs, every=60):
    # jobs: {name: (Popen, log path)}; prints progress, raises on failure
    while True:
        running = False
        for name, (proc, log) in jobs.items():
            text = Path(log).read_text(errors="ignore")
            code = proc.poll()
            if code is None:
                running = True
                print(f"[{(time.time() - T0) / 60:6.1f} min] {name}: {last_line(text)[:160] or 'working (prints when done)'}", flush=True)
            elif code != 0:
                print(text[-4000:])
                raise RuntimeError(f"{name} failed with exit code {code}; rerun this cell to resume")
        if not running:
            for name, (proc, log) in jobs.items():
                print(f"{name}: {last_line(Path(log).read_text(errors='ignore'))[:300]}")
            return
        time.sleep(every)

def script(name):
    return str(REPO / "scripts" / name)

gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()
print("\n".join(gpus))
assert len(gpus) == 2, "Set Accelerator to GPU T4 x2"
print("finished steps:", sorted(p.stem for p in STAGES.glob("*.done")) or "none")
T0 = time.time()

In [ ]:
# 2. Code, the English weights and the phoneme aligner (installed every session; downloads are kept)
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO)], check=True)
print("code:", subprocess.run(["git", "-C", str(REPO), "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "librosa", "soundfile", "pyarrow", "scipy"], check=True)
sys.path.insert(0, str(REPO / "src"))

from huggingface_hub import hf_hub_download, snapshot_download
WEIGHTS = WORK / "weights" / "tiny-voca.pt"
if not WEIGHTS.exists():
    WEIGHTS = Path(hf_hub_download("n4ze3m/tiny-voca", "tiny-voca.pt", local_dir=WORK / "weights"))
if not (done("split") and done("annotate")):
    snapshot_download("facebook/wav2vec2-lv-60-espeak-cv-ft")   # used only to align training data
print("English weights:", WEIGHTS)

In [ ]:
# 3. Check the Luganda front end (these are the tokens the released model was trained on)
check = subprocess.run([sys.executable, "-c",
    "from tiny_voca.text import phonemize\n"
    "for s in ['Oli otya? Nkwagala nnyo, ssebo.', \"Ng'ombe za kitaawe n'abaana bajja.\"]:\n"
    "    print(s, '->', ' '.join(phonemize(s).symbols))"], capture_output=True, text=True)
print(check.stdout, check.stderr[-3000:])
assert check.returncode == 0
assert "<bos> ɔ l i <sp> ɔ t j ɐ ? n k w ɐ ɡ ɐ l ɐ <sp> n n j ɔ , s s ɛ b ɔ . <eos>" in check.stdout, "front end changed"

In [ ]:
# 4. Export the two speakers (train + validation splits) at 24 kHz
import soundfile as sf
from IPython.display import Audio, display

if not skip("export"):
    exported = subprocess.run([sys.executable, script("export_waxal.py"), "--output", str(DATA),
                               "--female", FEMALE, "--male", MALE], capture_output=True, text=True)
    print(exported.stdout, exported.stderr[-3000:])
    assert exported.returncode == 0, "export failed; rerun this cell (finished WAV files are kept)"
    mark("export", **{voices[s]: len(read_jsonl(DATA / f"clips-{s}.jsonl")) for s in voices})

for s in voices:
    first = read_jsonl(DATA / f"clips-{s}.jsonl")[0]
    x, sr = sf.read(first["audio"])
    print(f"speaker {s} ({voices[s]}):")
    display(Audio(x[:8 * sr], rate=sr))

In [ ]:
# 5. Cut long clips into 4-18 s chunks at word pauses: one speaker per GPU (resumes clip by clip)
if not skip("split"):
    jobs = {}
    for gpu, s in enumerate(voices):
        out = DATA / f"chunks-{s}"
        if (out / "COMPLETE").exists():
            continue
        jobs[f"split-{s}"] = start(f"split-{s}", [sys.executable, script("split_clips.py"),
                                                  "--clips", str(DATA / f"clips-{s}.jsonl"), "--output", str(out)], gpu)
    watch(jobs)
    minutes = {voices[s]: round(sum(r["seconds"] for r in read_jsonl(DATA / f"chunks-{s}" / "chunks.jsonl")) / 60) for s in voices}
    mark("split", minutes=minutes)

In [ ]:
# 6. Choose the training audio, then acoustic features (sharded across CPU cores; each shard resumes)
if not skip("features"):
    if not (DATA / "train.jsonl").exists():   # written once, so a resumed run sees identical inputs
        chosen = subprocess.run([sys.executable, script("make_manifests.py"), "--data", str(DATA), "--female", FEMALE,
                                 "--male", MALE, "--minutes", str(MINUTES_PER_VOICE)], capture_output=True, text=True)
        print(chosen.stdout, chosen.stderr[-3000:])
        assert chosen.returncode == 0
    # One part per CPU core, so the parts can be prepared side by side
    if not any(DATA.glob("train-*.jsonl")):
        cores = os.cpu_count() or 1
        for split in ("train", "validation"):
            items = read_jsonl(DATA / f"{split}.jsonl")
            parts = max(1, min(cores, len(items) // 20)) if split == "train" else 1
            for k in range(parts):
                write_jsonl(DATA / f"{split}-{k}.jsonl", items[k::parts])
            print(split, len(items), "chunks in", parts, "shard(s)")

    jobs = {}
    for part in sorted(DATA.glob("train-*.jsonl")) + sorted(DATA.glob("validation-*.jsonl")):
        shard = FEATURES / "shards" / part.stem
        meta = shard / f"{part.stem.rsplit('-', 1)[0]}.preparation.json"
        if meta.exists() and json.loads(meta.read_text()).get("complete"):
            continue
        jobs[f"prepare-{part.stem}"] = start(f"prepare-{part.stem}", [sys.executable, "-m", "tiny_voca.prepare",
            "--inputs", str(part), "--split", part.stem.rsplit("-", 1)[0], "--output", str(shard), "--resume"],
            single_thread=True)
    print(f"{len(jobs)} feature jobs to run on {os.cpu_count()} CPU cores")
    watch(jobs, every=120)

    # One manifest per split, pointing into the shard folders (nothing is moved, so shards stay resumable)
    counts = {}
    for split in ("train", "validation"):
        merged = []
        for shard in sorted((FEATURES / "shards").glob(f"{split}-*")):
            for r in read_jsonl(shard / f"{split}.jsonl"):
                r["path"] = f"shards/{shard.name}/{r['path']}"
                merged.append(r)
            shutil.copy(shard / "features.json", FEATURES / "features.json")
        write_jsonl(FEATURES / f"{split}.jsonl", merged)
        (FEATURES / f"{split}.preparation.json").write_text(json.dumps({"complete": True, "accepted": len(merged)}))
        counts[split] = len(merged)
    mark("features", **counts)

In [ ]:
# 7. Phoneme durations by forced alignment (resumes from its journal)
from collections import Counter
if not skip("annotate"):
    command = [sys.executable, "-m", "tiny_voca.annotate",
               "--features", str(FEATURES), "--output", str(ANN), "--dataset-revision", "CraneAILabs/waxal-lug-clean",
               "--splits", "train", "validation", "--min-token-probability", "0.35", "--max-weak-phone-fraction", "0.5",
               "--max-phone-frames", "90"]
    if (ANN / "annotations.json").exists():
        command.append("--resume")
    watch({"annotate": start("annotate", command, 0)}, every=60)
    assert json.loads((ANN / "annotations.json").read_text())["complete"], "annotation run is not marked complete"
    report = {}
    for split in ("train", "validation"):
        ok, bad = read_jsonl(ANN / f"{split}.jsonl"), read_jsonl(ANN / f"{split}.rejected.jsonl")
        minutes = Counter()
        for r in ok:
            minutes[str(r.get("speaker"))] += r.get("frames", 0) * 256 / 24000 / 60
        report[split] = {"accepted": len(ok), "rejected": len(bad), "minutes": {k: round(v) for k, v in minutes.items()}}
        print(f"{split}: {report[split]}")
        print("  rejection reasons:", Counter(reason for r in bad for reason in r["reasons"]).most_common())
    mark("annotate", **report)

In [ ]:
# 8. Train ONE model on both T4s (data-parallel via torchrun); resumes from run/last.pt
import torch
TRAIN_SETTINGS = LOGS / "train-settings.json"   # outside run/: the trainer needs an empty folder to start

def saved_step(path):
    return torch.load(path, map_location="cpu", weights_only=True)["step"] if path.exists() else 0

def launch(batch_frames, seconds, resume):
    if not resume:
        shutil.rmtree(RUN, ignore_errors=True)
    cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node", "2",
           "-m", "tiny_voca.train",
           "--train", str(ANN / "train.jsonl"), "--validation", str(ANN / "validation.jsonl"),
           "--output", str(RUN), "--init-weights", str(WEIGHTS),
           "--steps", str(STEPS), "--max-seconds", str(int(seconds)),
           "--learning-rate", str(LEARNING_RATE), "--warmup", "300",
           "--gan-start", "1000", "--gan-ramp", "1000",
           "--teacher-hold", "500", "--teacher-ramp", "1500", "--teacher-final", "0",
           "--batch-frames", str(batch_frames), "--save-every", "500", "--keep-every", "1000", "--log-every", "50"]
    if resume:
        cmd += ["--resume", str(RUN / "last.pt")]
    log = LOGS / "train.log"
    with open(log, "a") as stream:
        stream.write(f"\n=== {'resume' if resume else 'start'} at {time.ctime()} (batch frames {batch_frames}) ===\n")
    proc = subprocess.Popen(cmd, env=dict(os.environ), stdout=open(log, "a"), stderr=subprocess.STDOUT)
    if not resume:
        TRAIN_SETTINGS.write_text(json.dumps({"batch_frames": batch_frames}))
    return proc, log

def progress(text):
    text = text.rsplit("\n=== ", 1)[-1]   # this launch only
    for line in reversed(text.splitlines()):
        if line.startswith("{") and '"steps_per_second"' in line:
            try:
                return json.loads(line)
            except json.JSONDecodeError:
                pass
    return None

if not skip("train"):
    step = saved_step(RUN / "last.pt")
    if step >= STEPS:
        mark("train", step=step)
    else:
        resume = step > 0
        batch_frames = json.loads(TRAIN_SETTINGS.read_text())["batch_frames"] if resume else BATCH_FRAMES
        print(f"resuming at step {step}" if resume else "starting from the English weights")
        began = time.time()
        proc, log = launch(batch_frames, MAX_TRAIN_SECONDS, resume)
        while True:
            time.sleep(120)
            code, text = proc.poll(), log.read_text(errors="ignore")
            r = progress(text)
            if r:
                left = (STEPS - r["step"]) / max(r["steps_per_second"], 1e-9) / 60
                print(f"[{(time.time() - began) / 60:5.1f} min] step {r['step']}/{STEPS}  {r['steps_per_second']:.2f} steps/s  "
                      f"loss {r['loss']:.3f}  ~{left:.0f} min left", flush=True)
            if code is None:
                continue
            last = text.rsplit("\n=== ", 1)[-1]
            if code != 0 and "out of memory" in last and batch_frames > 6000 and saved_step(RUN / "last.pt") == 0:
                batch_frames = 6000
                print("A T4 ran out of memory before the first save: restarting with --batch-frames 6000")
                proc, log = launch(batch_frames, MAX_TRAIN_SECONDS - (time.time() - began), False)
                continue
            if code != 0:
                print(last[-5000:])
                raise RuntimeError(f"training failed with exit code {code}; rerun this cell to resume from the last save")
            break
        step = saved_step(RUN / "last.pt")
        if step >= STEPS:
            mark("train", step=step, minutes=round((time.time() - began) / 60))
        else:
            print(f"Stopped at step {step}/{STEPS} by the time cap: run this cell again (or the next session) to continue.")
print("kept checkpoints:", sorted(p.name for p in RUN.glob("step-*.pt")))

In [ ]:
# 9. Listen to the last checkpoints (both voices, held-out sentences) and pick one
from tiny_voca.text import set_frontend
set_frontend("lug")
from tiny_voca.infer import load_model, speak_text

CKPT = OUT / "checkpoints"
CKPT.mkdir(exist_ok=True)
candidates = {}
for state in sorted(RUN.glob("step-*.pt"))[-3:]:          # e.g. steps 2000, 3000, 4000
    target = CKPT / f"{state.stem}.pt"
    if not target.exists():
        subprocess.run([sys.executable, script("export_checkpoint.py"),
                        "--checkpoint", str(state), "--output", str(target)], check=True)
    candidates[state.stem] = target
final_step = saved_step(RUN / "inference.pt")
if f"step-{final_step:08d}" not in candidates:             # stopped between kept steps
    candidates[f"step-{final_step:08d}"] = RUN / "inference.pt"

sentences = [s for s in (DATA / "test_sentences.txt").read_text(encoding="utf-8").splitlines() if s.strip()]
for name, checkpoint in candidates.items():
    model = load_model(checkpoint)
    for voice in ("F2", "M2"):
        for i, sentence in enumerate(sentences[:2]):
            path = OUT / "samples" / name / f"{voice}-{i}.wav"
            if not path.exists():
                wave, _ = speak_text(model, sentence, voice)
                path.parent.mkdir(parents=True, exist_ok=True)
                sf.write(path, wave.detach().cpu().numpy(), model.config.sample_rate)
            print(f"{name} | {voice} | {sentence}")
            display(Audio(str(path)))

In [ ]:
# 10. Package the chosen checkpoint in /kaggle/working/luganda/model (PyTorch + ONNX + GGUF)
BEST = None   # after listening, e.g. "step-00003000"; None keeps the last checkpoint
best = BEST or list(candidates)[-1]
final = OUT / "model"
final.mkdir(exist_ok=True)
shutil.copy(candidates[best], final / "tiny-voca-lug.pt")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"], check=False)
onnx = subprocess.run([sys.executable, "-m", "tiny_voca.export", "--checkpoint", str(final / "tiny-voca-lug.pt"),
                       "--output", str(final / "model.onnx")], capture_output=True, text=True)
print(onnx.stdout[-1500:], onnx.stderr[-1500:])
gguf = subprocess.run([sys.executable, script("export_gguf.py"), "--checkpoint", str(final / "tiny-voca-lug.pt"),
                       "--output-dir", str(final / "gguf")], capture_output=True, text=True)
print(gguf.stdout[-1500:], gguf.stderr[-1500:])
mark("package", checkpoint=best)
print("model folder:", sorted(p.name for p in final.iterdir()))

# Optional: push to Hugging Face (add an HF_TOKEN secret under Add-ons -> Secrets first).
# The model card (README.md) is kept on Hugging Face; upload_folder leaves it as it is.
# from kaggle_secrets import UserSecretsClient
# from huggingface_hub import HfApi
# api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
# api.create_repo("YOUR_NAME/tiny-voca-lg", exist_ok=True)
# api.upload_folder(folder_path=str(final), repo_id="YOUR_NAME/tiny-voca-lg")